# Silver Layer — Clean Data & Feature Extraction

Welcome to the **Silver Layer**. This is where raw pixels become structured game state.

## Where Are We in the Pipeline?

```
Raw Screenshot  ──►  [BRONZE]  ──►  [SILVER]  ──►  [GOLD]  ──►  Prediction
                      (denoise,      (you are here)  (train model)
                       sharpen)
```

The Bronze layer gave us a clean image. Now we need to **extract structured,
game-relevant features** from that image — things like:

- **Player health** (how hurt is the player?)
- **Enemy positions and health** (where are enemies, how hurt are they?)
- **Attack state** (is the player attacking?)
- **Defense state** (is the player blocking/dodging?)
- **Damage indicators** (is the screen showing damage feedback?)

These features are fed to the **Gold layer** (a classifier) which produces the
final `winning`/`losing`/`stalemate` prediction.

---

## Why a CNN Instead of Hardcoded Positions?

A naive approach would be to hardcode pixel coordinates:
```python
player_health_bar = image[20:50, 2000:2500]  # brittle!
```

This breaks if:
- The game UI gets updated
- You switch to a different game
- The resolution changes
- Enemies move to unexpected positions

**Instead, we use a CNN that learns the relationships end-to-end:**
- The CNN sees the whole image
- It learns what a health bar *looks like*, not where it *is*
- It learns that enemy health bars are *above enemy heads*
- It learns that a red flash means *player damage*

This makes the system **game-adaptive** — retrain on new labeled data and it
works with any game.

---

## Setup: Imports & Paths

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import random
import tempfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch

from src.config import BRONZE_DIR, SILVER_DIR
from src.pipeline.bronze import load_image, preprocess, process_image as bronze_process
from src.pipeline.silver import (
    H, W,
    SilverFeatures,
    process_image as silver_process,
    render_synthetic_frame,
    generate_synthetic_annotation,
)
from src.models.silver_cnn import SilverCNN, SilverOutput, SilverDataset

sns.set_theme(style="darkgrid")
%matplotlib inline

---

# Part 1: Understanding the Multi-Head CNN

## Model Architecture

The `SilverCNN` is a **multi-head CNN** — one shared backbone with several
specialized output heads:

```
                  ┌─── player_health ──────── (scalar 0..1)
                  ├─── player_position ────── (cx, cy normalized)
     ┌────────┐   ├─── enemy_heatmap ──────── (3 × 16 × 28 heatmaps)
     │Backbone│───├─── enemy_health ───────── (3 scalars 0..1)
     └────────┘   ├─── attacking ──────────── (binary 0..1)
                  ├─── defending ──────────── (binary 0..1)
                  └─── damage_indicator ───── (binary 0..1)
```

### Backbone (shared feature extractor)
6 convolutional blocks that progressively downsample the 1440×2560 input
to a 23×40 feature map (stride 64). Every block is Conv → BatchNorm → ReLU.

### Why multiple heads?
Each head learns different features from the same backbone:
- **Player health** needs to find the health bar color/length
- **Enemy heatmap** needs to detect character-shaped blobs
- **Attacking/defending** needs to recognize motion cues and pose

Training all heads jointly allows them to share low-level features
(edges, colors, textures) while specializing at the output.

### Heatmap approach for enemy detection
Instead of bounding box regression, the model outputs **heatmaps** —
2D probability maps at 16×28 resolution. Each of the 3 channels (for up to
3 enemies) peaks at the enemy's location. This is:
- **Faster** than region proposal networks
- **Simple** to train (just MSE on the heatmap)
- **Flexible** — heatmaps can represent fuzzy spatial concepts

In [ ]:
model = SilverCNN()
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

## Test a Forward Pass

Feed a random image through the model to verify the output shapes and ranges.

In [ ]:
x = torch.randn(1, 3, H, W)
with torch.no_grad():
    out = model(x)

print("Output shapes:")
print(f"  player_health:    {out.player_health.shape}")
print(f"  player_position:  {out.player_position.shape}")
print(f"  enemy_heatmap:    {out.enemy_heatmap.shape}")
print(f"  enemy_health:     {out.enemy_health.shape}")
print(f"  attacking:        {out.attacking.shape}")
print(f"  defending:        {out.defending.shape}")
print(f"  damage_indicator: {out.damage_indicator.shape}")

print("\nOutput ranges (all 0..1):")
print(f"  player_health:    {out.player_health.item():.4f}")
print(f"  player_position:  ({out.player_position[0,0]:.4f}, {out.player_position[0,1]:.4f})")
print(f"  enemy_health:     {out.enemy_health[0].tolist()}")
print(f"  attacking:        {out.attacking.item():.4f}")
print(f"  defending:        {out.defending.item():.4f}")
print(f"  damage_indicator: {out.damage_indicator.item():.4f}")

heatmap_np = out.enemy_heatmap[0, 0].cpu().numpy()
plt.figure(figsize=(6, 4))
plt.imshow(heatmap_np, cmap="hot", aspect="auto")
plt.colorbar(label="Enemy presence probability")
plt.title("Sample Enemy Heatmap (Channel 0)")
plt.xlabel("Width (28 cells)")
plt.ylabel("Height (16 cells)")
plt.tight_layout()
plt.show()

---

# Part 2: Synthetic Data Generation

To train the SilverCNN, we need labeled data. The
`render_synthetic_frame()` and `generate_synthetic_annotation()` functions
create this data programmatically. Each synthetic frame contains:
- A **player** circle at center screen
- **1-3 enemy** circles at random positions
- **Health bars** above each enemy's head and in the top-right for the player

This lets us **test the training pipeline** before collecting real labeled
screenshots.

In [ ]:
synthetic_img = render_synthetic_frame(num_enemies=2, seed=42)
synthetic_ann = generate_synthetic_annotation(num_enemies=2, seed=42)

print("Synthetic Annotation:")
print(json.dumps(synthetic_ann, indent=2))

fig, ax = plt.subplots(1, 1, figsize=(16, 9))
ax.imshow(synthetic_img)
ax.set_title("Synthetic Training Frame\n(Green = player, Red = enemies)", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

## Generating a Dataset

The `SilverDataset` class pairs images and annotations for PyTorch training.

In [ ]:
tmp_dir = Path(tempfile.mkdtemp())
img_dir = tmp_dir / "images"
ann_dir = tmp_dir / "annotations"
img_dir.mkdir(exist_ok=True)
ann_dir.mkdir(exist_ok=True)

for i in range(5):
    img = render_synthetic_frame(seed=i)
    ann = generate_synthetic_annotation(seed=i)
    cv2.imwrite(str(img_dir / f"frame_{i:03d}.png"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    with open(ann_dir / f"frame_{i:03d}.json", "w") as f:
        json.dump(ann, f)

ds = SilverDataset(str(tmp_dir))
print(f"Dataset length: {len(ds)}")

x_sample, y_sample = ds[0]
print(f"Image tensor shape: {x_sample.shape}")
print(f"Player health label: {y_sample.player_health.item():.4f}")
print(f"Attacking label: {y_sample.attacking.item()}")
print(f"Defending label: {y_sample.defending.item()}")

import shutil
shutil.rmtree(tmp_dir)

---

# Part 3: Running the Silver Pipeline on Real Screenshots

Now let's process a real screenshot through the Silver layer.
Without a trained model, defaults are returned.

In [ ]:
pngs = sorted(BRONZE_DIR.glob("*.png"))
if not pngs:
    raise FileNotFoundError(f"No PNGs found in {BRONZE_DIR}")

real_image_path = str(pngs[0])
print(f"Processing: {pngs[0].name}")

bronze_result = bronze_process(real_image_path, str(SILVER_DIR))
print(f"Bronze output: {bronze_result['preprocessed']}")

silver_result = silver_process(real_image_path, str(SILVER_DIR))
print(f"\nSilver features saved to: {silver_result['features_json']}")

In [ ]:
features = silver_result["silver_features"]

print("=" * 50)
print("SILVER FEATURES")
print("=" * 50)
print(f"Image path         : {features['image_path']}")
print(f"Player health      : {features['player_health']:.3f}")
print(f"Player position    : ({features['player_position'][0]:.3f}, {features['player_position'][1]:.3f})")
print(f"Number of enemies  : {features['num_enemies']}")
print(f"Attacking          : {features['attacking']}")
print(f"Defending          : {features['defending']}")
print(f"Damage indicator   : {features['damage_indicator']}")

if features['enemies']:
    print("\nEnemy Details:")
    for i, enemy in enumerate(features['enemies']):
        print(f"  Enemy {i+1}: bbox={enemy['bbox']}, health={enemy['health']:.3f}")
else:
    print("\n(No enemies detected — model not trained yet)")

## Visualize the Silver Features

Overlay extracted features back onto the screenshot.

In [ ]:
def draw_silver_features(image, features):
    vis = image.copy()
    cx = int(features['player_position'][0] * W)
    cy = int(features['player_position'][1] * H)
    cv2.circle(vis, (cx, cy), 50, (0, 255, 0), 3)
    cv2.putText(vis, "PLAYER", (cx - 40, cy - 60),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    health_pct = features['player_health'] * 100
    cv2.putText(vis, f"Health: {health_pct:.0f}%", (W - 550, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    for i, enemy in enumerate(features['enemies']):
        bx, by, bw, bh = enemy['bbox']
        cv2.rectangle(vis, (bx, by), (bx + bw, by + bh), (255, 0, 0), 3)
        cv2.putText(vis, f"ENEMY {i+1}", (bx, by - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
        hx, hy, hw, hh = enemy['health_bar_bbox']
        fill = int(hw * enemy['health'])
        cv2.rectangle(vis, (hx, hy), (hx + hw, hy + hh), (100, 100, 100), -1)
        color = (0, 255, 0) if enemy['health'] > 0.5 else (0, 0, 255)
        cv2.rectangle(vis, (hx, hy), (hx + fill, hy + hh), color, -1)
    y_offset = 120
    if features['attacking']:
        cv2.putText(vis, "STATE: ATTACKING", (W - 550, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 165, 255), 2); y_offset += 40
    if features['defending']:
        cv2.putText(vis, "STATE: DEFENDING", (W - 550, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2); y_offset += 40
    if features['damage_indicator']:
        cv2.putText(vis, "DAMAGE TAKEN", (W - 550, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
    return vis

bronze_img = cv2.imread(bronze_result['preprocessed'])
bronze_img = cv2.cvtColor(bronze_img, cv2.COLOR_BGR2RGB)
vis = draw_silver_features(bronze_img, features)

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
axes[0].imshow(bronze_img)
axes[0].set_title("Bronze (Preprocessed)", fontsize=14, fontweight="bold")
axes[0].axis("off")
axes[1].imshow(vis)
axes[1].set_title("Silver — Extracted Features Overlay", fontsize=14, fontweight="bold")
axes[1].axis("off")
plt.tight_layout()
plt.show()

---

# Part 4: Training with MLflow

The Silver training pipeline is integrated with **MLflow** for experiment
tracking. Every run logs:
- **Parameters**: learning rate, batch size, epochs, model architecture
- **Metrics**: training and validation loss (logged every epoch)
- **Artifacts**: trained model, sample predictions

## Quick Start: `train_silver_model()`

Pass a path to a dataset directory (containing ``images/`` and
``annotations/``):

```python
from src.pipeline.silver_train import train_silver_model

result = train_silver_model(
    data_root="path/to/dataset",
    learning_rate=1e-4,
    batch_size=2,
    num_epochs=50,
    experiment_name="silver_cnn",
)
print(f"Run ID: {result['run_id']}, Best val_loss: {result['best_val_loss']:.4f}")
```

## Or Generate Synthetic Data First

If you don't have real labeled data yet, use `generate_and_train()` to
create a synthetic dataset and train in one call:

In [ ]:
from src.pipeline.silver_train import generate_and_train

result = generate_and_train(
    synthetic_samples=30,
    learning_rate=1e-4,
    batch_size=2,
    num_epochs=5,
    experiment_name="silver_cnn_demo",
    run_name="demo_run",
)

print(f"\nRun ID: {result['run_id']}")
print(f"Best validation loss: {result['best_val_loss']:.6f}")

## What MLflow Tracks

Run `mlflow ui` in the project root to open the MLflow dashboard at
http://localhost:5000. You will see:

### Parameters (logged once)
```
learning_rate:     0.0001
batch_size:        2
num_epochs:        50
val_split:         0.15
train_samples:     25
val_samples:       5
model:             SilverCNN
backbone_channels: [32, 64, 128, 256, 384, 512]
```

### Metrics (logged every epoch)
```
train_loss:  ... (decreasing over time)
val_loss:    ... (should also decrease; if it diverges, you're overfitting)
```

### Artifacts
- **model/** — the full PyTorch model saved with MLflow's PyTorch flavor
- **sample_predictions.txt** — predicted vs target values for validation samples
- **checkpoints/** — best model weights (lowest val_loss)

## The Multi-Task Loss

All heads are trained simultaneously with a combined loss:

```math
L = MSE(player_health) + MSE(player_position) +
    MSE(enemy_heatmap) + MSE(enemy_health) +
    BCE(attacking) + BCE(defending) + BCE(damage)
```

MSE for regression, BCE for classification. All weighted equally.

---

# Part 5: Data Requirements

For a **semi-viable model** (~75–85% reliability on common cases):

| Task | Min Samples | Why This Many |
|------|-------------|---------------|
| Player health | 200–300 | Single scalar, visually salient |
| Enemy detection | 500–1000 | 1–3 enemies, varying positions |
| Enemy health | 500–1000 | Tied to enemy detection |
| Attacking/defending | 300–500 each | Needs balanced classes |
| Damage indicator | 200–300 | Red flash is visually distinct |

**Total recommended: 1000–2000 labeled screenshots**

This assumes accurate annotations and a representative distribution of game
states. Fewer samples risks the model memorizing rather than generalizing.

**Synthetic data** validates the pipeline but cannot replace real data —
synthetic frames lack the texture and variation of real game screenshots.
Start with ~500 synthetic samples to test, then collect 1000+ real labels.

---

# Part 6: Running with a Trained Model

To load a trained checkpoint and run inference:

In [ ]:
# After training, load the best model and process an image:
#
# from src.pipeline.silver_train import train_silver_model
# from src.models.silver_cnn import SilverCNN
#
# result = train_silver_model(data_root="...", num_epochs=100)
# model = SilverCNN()
# model.load_state_dict(torch.load("checkpoints/best_model.pth"))
#
# # Now use the trained model for extraction
# silver_result = silver_process("screenshot.png", "data/silver/", model=model)

print("See Part 4 for training. Then pass model= to silver_process().")

---

# Part 7: Going Deeper — The Design Document

For a complete, **code-free** walkthrough of the architecture and design
decisions, read the design document:

**`docs/design.md`**

It covers:
- Why three stages (Bronze → Silver → Gold) instead of one end-to-end model
- Why a shared backbone with multiple heads
- Why heatmaps instead of bounding boxes for enemy detection
- The enemy-to-health-bar spatial relationship
- The trade-offs behind every major design choice
- Common failure modes and how to diagnose them

This document helps you **internalize** the system — the notebooks show you
what the code does, and the design doc tells you *why* it was built that way.

---

## Summary

- **Multi-head CNN** with 7 output heads for game feature extraction
- **MLflow** tracks every training run (params, metrics, artifacts)
- **Synthetic data** enables pipeline testing before real labels exist
- **1000–2000 labeled samples** recommended for a semi-viable model
- **docs/design.md** explains the architectural thinking in plain text

Next: open **`03_gold_modeling.ipynb`** to train the final classifier.